In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [ ]:
load_dotenv()
openai_client = OpenAI()

In [ ]:
documents = load_faq_data()
index = build_index(documents)

In [ ]:

INSTRUCTIONS = """
You are a question-answering assistant for the LLM Zoomcamp course.

Your task is to answer questions using ONLY the information
provided in the context.

Rules:
1. Treat the context as the only source of truth.
2. Do not use your pretrained knowledge or external knowledge.
3. Do not make assumptions or invent information.
4. If the context does not contain enough information to answer
   the question, respond exactly with:
   I don't know
5. Do not answer questions unrelated to the course.
6. If the answer is available in the context, provide a clear
   and concise answer based on that information.
""".strip()

In [ ]:
assistant = RAGBase(
    index = index,
    instructions=INSTRUCTIONS,
    llm_client=openai_client
)

In [ ]:
answer  = assistant.rag("How to install Ollama locally?")
print(answer)

In [ ]:
def search(query):
    boost_dict = {'question': 3.0, 'section' : 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results = 5,
        boost_dict = boost_dict,
        filter_dict = filter_dict
    )

In [ ]:
search_tool = {
    "type": "function",
    "name":"search",
    "description": 'Search the FAQ database for entries matching the given query',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                'type': 'string',
                'description': 'Seacrh query text to look up in the course FAQ'
            }
        },
        "required":["query"],
        'additionalProperties': False
    }
}

In [ ]:
messages = [{'role': 'user', 'content': ' I just discovered the course. Can I join it?'}]
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

call = response.output[0]

In [ ]:
import json

In [ ]:
args = json.loads(call.arguments)
args

In [ ]:
results = search(**args)
result_json = json.dumps(results)
print(result_json)

In [ ]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="I just discovered the course. Can I join it?",
    tools=[search_tool]
)

print(response.output)

call = response.output[0]

result_json = json.dumps(results)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    previous_response_id=response.id,
    input=[function_call_output],
    tools=[search_tool]
)

print(response.output)

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question ='I just discovered the course. Can I join?'

messages = [{
    'role': 'developer', 'content': instructions},
    {'role':'user', 'content': question}
]

In [ ]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)
response.output

In [ ]:

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    else:
        raise ValueError(f"Unknown function: {call.name}")

    # Convert the result into a JSON string
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input="I just discovered the course. Can I join it?",
    tools=[search_tool]
)

messages = []

messages.extend(response.output)

for item in response.output:
    print(item)

    if item.type == "function_call":
        function_call_output = make_call(item)
        messages.append(function_call_output)

# Send the function result back to the model
if messages:
    final_response = openai_client.responses.create(
        model="gpt-5.4-mini",
        previous_response_id=response.id,
        input=messages[len(response.output):],
        tools=[search_tool]
    )

    print(final_response.output)


In [ ]:
question =  " I just discoevered the course. Can I join?"
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool],
)

messages.extend(response.output)
has_function_calls = False
for item in response.output:
    if item.type =="function_call":
        print("fubction_call", item.name, item.arguments)
        call_output = make_call(item)
        has_function_calls = True
    elif item.type =="message":
        print("Asstant: ")
        print(item.content[0].text)

In [ ]:
it = 1
while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)
    for item in response.output:
        if item.type =="function_call":
            print("fubction_call", item.name, item.arguments)
            call_output = make_call(item)
            has_function_calls = True
        elif item.type =="message":

            print("Asstant: ")
            print(item.content[0].text)

    it=+1
    if has_function_calls ==False:
        break

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")